In [10]:
#defining the node class to  store information about each split 

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature 
        self.threshold = threshold 
        self.left = left 
        self.right = right 
        self.value = value 

    def is_leaf_node(self):
        return self.value is not None 

In [11]:
# Math functions for entropy and information gain 

import numpy as np 

def entropy(y):
    hist = np.bincount(y)
    ps = hist/len(y) 

    # H = -sum(p*log2(p)) 
    return -np.sum([p * np.log2(p) for p in ps if p > 0]) 

def information_gain(y, X_column, threshold):
    parent_entropy = entropy(y)

    left_idxs = np.argwhere(X_column <= threshold).flatten() 
    right_idxs = np.argwhere(X_column > threshold).flatten() 

    if len(left_idxs) == 0 or len(right_idxs) == 0:
        return 0
    
    n=len(y) 
    n_l, n_r = len(left_idxs), len(right_idxs) 
    e_l, e_r = entropy(y[left_idxs]), entropy(y[right_idxs])
    child_entropy = (n_l / n)* e_l + (n_r / n)* e_r 

    #information gain = parentEntropy - childEntropy 
    return parent_entropy - child_entropy 

In [13]:
# Defining the decision tree class to build the tree 

class DecisionTree:
    def __init__(self, max_depth = 10, min_samples_split = 2):
        self.max_depth = max_depth 
        self.min_samples_split = min_samples_split 
        self.root = None 

    def fit(self, X, y):
        self.root = self._grow_tree(X, y) 

    def _grow_tree(self, X, y, depth = 0):
        n_samples, n_features = X.shape 
        n_labels = len(np.unique(y)) 

        #Stopping criteria 
        if(depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            leaf_value = self._most_common_label(y) 
            return Node(value = leaf_value) 
        
        #Find the best split 
        feat_idxs = np.arange(n_features) 
        best_feat, best_thresh = self._best_split(X, y, feat_idxs)

        #Create child Nodes 
        left_idxs = np.argwhere(X[:, best_feat] <= best_thresh).flatten()
        right_idxs = np.argwhere(X[:, best_feat] > best_thresh).flatten() 

        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth+1) 
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1) 
        return Node(best_feat, best_thresh, left, right) 
    
    def _best_split(self, X, y, feat_idxs):
        best_gain = -1
        split_idx, split_threshold = None, None 

        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx] 
            thresholds = np.unique(X_column) 

            for thr in thresholds:
                gain = information_gain(y, X_column, thr) 
                if gain > best_gain:
                    best_gain = gain 
                    split_idx = feat_idx 
                    split_threshold = thr 

        return split_idx, split_threshold 
    
    def _most_common_label(self, y):
        return np.bincount(y).argmax() 
    
    # defining functions to predict and traverse trees 

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X]) 
    
    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value 
        
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.right) 
        return self._traverse_tree(x, node.right) 
    

if __name__ == "__main__" :

    # Features: [Hours Studied, Hours Slept]
    X = np.array([
        [1, 5], [2, 6], [3, 4], [7, 8], [8, 9], [9, 7], [2, 2], [1, 1], [6, 5], [7, 3]
    ])

    # Labels: 1 = Pass, 0 = Fail
    y = np.array([0, 0, 0, 1, 1, 1, 0, 0, 1, 1])

    clf = DecisionTree(max_depth=3)
    clf.fit(X, y)

    # Testing on a new student: 8 hours study, 2 hours sleep
    test_student = np.array([[0, 2]])
    prediction = clf.predict(test_student)

    print(f"Prediction for [2 hrs study, 2 hrs sleep]: {'Pass' if prediction[0] == 1 else 'Fail'}")

Prediction for [2 hrs study, 2 hrs sleep]: Pass
